In [14]:
from dotenv import load_dotenv
from env_utils import doublecheck_env,doublecheck_pkgs

# Load environment variables from .env file

load_dotenv()

#Check environment variables and packages
doublecheck_env(".env.example")
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)

OPENROUTER_API_KEY=****bfed
MODEL=****free
LANGSMITH_API_KEY=<not set>
LANGSMITH_TRACING=<not set>
LANGSMITH_PROJECT=<not set>
Python 3.11.9 satisfies requires-python: >=3.11
package              | required | installed | status | path                                                           
-------------------- | -------- | --------- | ------ | ---------------------------------------------------------------
langchain            | >=0.3    | 1.3.14    | ✅ OK   | c:\Users\himan\Desktop\Langchain-agents\.venv\Lib\site-packages
langchain-core       | >=0.3    | 1.5.1     | ✅ OK   | c:\Users\himan\Desktop\Langchain-agents\.venv\Lib\site-packages
langchain-community  | >=0.3    | 0.4.2     | ✅ OK   | c:\Users\himan\Desktop\Langchain-agents\.venv\Lib\site-packages
langchain-openai     | >=0.2    | 1.4.1     | ✅ OK   | c:\Users\himan\Desktop\Langchain-agents\.venv\Lib\site-packages
langchain-openrouter | >=0.2.7  | 0.2.7     | ✅ OK   | c:\Users\himan\Desktop\Langchain-agents\.venv\Lib\site-p

In [15]:
# Build  a SQl agents
#import the SQLDatabase class from langchain_community.utilities and create an instance of it using the from_uri method, passing in the URI for the SQLite database "Chinook.db". This will allow you to interact with the database using SQL queries.
from langchain_community.utilities import SQLDatabase

# loading the SQLite database "Chinook.db" using the SQLDatabase class from langchain_community.utilities
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [16]:
# Defining the runtime context to provide the agent and tool with acces to the database

from dataclasses import dataclass

from langchain_community.utilities import SQLDatabase

#define context structure to support the dependency injection of the database into the agent and tool. This will allow the agent and tool to access the database and perform SQL queries as needed.
@dataclass
class RuntimeContext:
    db: SQLDatabase


In [17]:
from  langchain_core.tools import Tool, tool
from langgraph.runtime import get_runtime


# Define a custom tool that will allow the agent to execute SQL queries against the database. The tool will take a query string as input and return the results of the query.

@tool
def execute_sql_query(query: str) -> str:
    """" Ececute a SQL query against the database and return the results as a string."""
    
    # Get the runtime context and access the database instance  
    runtime=get_runtime(RuntimeContext)
    db=runtime.context.db
    
    # Execute the SQL query using the database instance and return the results as a string. If an error occurs during query execution, return an error message.
    try:
        return db.run(query)
    except Exception as e:
            return f"Error executing query: {e}"

In [ ]:


SYSTEM_PROMPT ="""You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [21]:
# Create agent using the SQLDatabase instance and the SYSTEM_PROMPT defined above. The agent will use the execute_sql_query tool to interact with the database and retrieve data as needed.

import os
from langchain.agents import create_agent
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

agent = create_agent(
    model=model,
    tools=[execute_sql_query],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

ValueError: Unable to infer model provider for model='openrouter/nvidia/nemotron-3-ultra-550b-a55b:free'. Please specify 'model_provider' directly.

Supported providers: anthropic, anthropic_bedrock, azure_ai, azure_openai, baseten, bedrock, bedrock_converse, cohere, deepseek, fireworks, google_anthropic_vertex, google_genai, google_vertexai, groq, huggingface, ibm, litellm, meta, mistralai, nvidia, ollama, openai, openrouter, perplexity, together, upstage, xai

For help with specific providers, see: https://docs.langchain.com/oss/python/integrations/providers

In [ ]:
from Ipython.display import display,I
